# Penerapan ADIWAKO untuk Optimasi Hyperparameter SVM
**Rosa Nur Aliana Sawafi — 2311110008**
Program Studi S1 Sains Data, Universitas Telkom Purwokerto

---
## Alur Eksperimen
1. Implementasi ADIWAKO & PSO Standar (dengan perbaikan stabilitas)
2. Validasi dengan Benchmark Functions Baru (Naser et al., 2024)
3. Eksperimen SVM Baseline, SVM+PSO, SVM+ADIWAKO
4. Evaluasi & Perbandingan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
from sklearn.datasets import load_breast_cancer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Library berhasil diimport!')

---
## BAGIAN 1: Implementasi ADIWAKO & PSO Standar

### Formula ADIWAKO (Sekyere et al., 2024)
- `mu    = (pbest_cost - gbest_cost) / |pbest_cost|`  ← pakai nilai absolut penyebut
- `delta = W_max - (W_max - W_min) * t / T_max`
- `w     = mu * tanh(delta)`
- `psi   = C_max - (C_max - C_min) * t / T_max`
- `c1 = c2 = mu * cosh(psi)`

### Perbaikan Stabilitas
- Penyebut mu menggunakan **nilai absolut** agar stabil untuk cost negatif
- mu di-clip ke [0, 1] agar w dan c tidak meledak
- Kecepatan partikel di-clamp agar tidak divergen

In [ ]:
class ADIWAKO:
    """
    Adaptive Dynamic Inertia Weight and Acceleration Coefficients PSO
    Sekyere et al. (2024) — dengan perbaikan stabilitas numerik.
    """
    def __init__(self, n_particles=30, max_iter=100,
                 W_max=1.0, W_min=0.1, C_max=5.0, C_min=2.0, bounds=None):
        self.n_particles = n_particles
        self.max_iter    = max_iter
        self.W_max = W_max;  self.W_min = W_min
        self.C_max = C_max;  self.C_min = C_min
        self.bounds = bounds

    def _adaptive_params(self, t, pbest_costs, gbest_cost):
        """
        Hitung w dan c adaptif per partikel.

        Perbaikan dari versi sebelumnya:
        - Penyebut mu pakai |pbest_cost| bukan pbest_cost
          → stabil untuk nilai cost negatif (Styblinski-Tang, Michalewicz)
        - mu di-clip ke [0, 1]
          → mencegah w dan c meledak ketika selisih cost sangat besar
        """
        # mu: adaptive factor — Persamaan (4) Sekyere et al.
        denom = np.abs(pbest_costs)
        denom = np.where(denom < 1e-10, 1e-10, denom)
        mu    = (pbest_costs - gbest_cost) / denom
        mu    = np.clip(mu, 0.0, 1.0)          # stabilisasi

        # delta: time-varying inertia component — Persamaan (5)
        delta = self.W_max - (self.W_max - self.W_min) * t / self.max_iter

        # w: adaptive dynamic inertia weight — Persamaan (3)
        w = mu * np.tanh(delta)                # w in [0, tanh(W_max)]

        # psi: time-varying acceleration component — Persamaan (7)
        psi = self.C_max - (self.C_max - self.C_min) * t / self.max_iter

        # c1 = c2: adaptive acceleration coefficients — Persamaan (6)
        c = mu * np.cosh(psi)                  # c1 = c2

        return w, c

    def optimize(self, fitness_func, n_dim):
        bounds = self.bounds or [(-100, 100)] * n_dim
        lb = np.array([b[0] for b in bounds])
        ub = np.array([b[1] for b in bounds])

        # Inisialisasi
        pos   = lb + np.random.rand(self.n_particles, n_dim) * (ub - lb)
        vel   = np.zeros((self.n_particles, n_dim))
        v_max = 0.2 * (ub - lb)              # velocity clamping

        costs = np.array([fitness_func(pos[i]) for i in range(self.n_particles)])

        pbest_pos  = pos.copy()
        pbest_cost = costs.copy()
        gbest_idx  = np.argmin(pbest_cost)
        gbest_pos  = pbest_pos[gbest_idx].copy()
        gbest_cost = pbest_cost[gbest_idx]
        history    = [gbest_cost]

        for t in range(1, self.max_iter + 1):
            w, c = self._adaptive_params(t, pbest_cost, gbest_cost)
            w_col = w.reshape(-1, 1)
            c_col = c.reshape(-1, 1)

            r1 = np.random.rand(self.n_particles, n_dim)
            r2 = np.random.rand(self.n_particles, n_dim)

            # Pembaruan kecepatan — Persamaan (15) proposal
            vel = (w_col * vel
                   + c_col * r1 * (pbest_pos - pos)
                   + c_col * r2 * (gbest_pos - pos))

            # Velocity clamping — mencegah divergensi
            vel = np.clip(vel, -v_max, v_max)

            # Pembaruan posisi
            pos = np.clip(pos + vel, lb, ub)

            # Evaluasi fitness
            costs = np.array([fitness_func(pos[i]) for i in range(self.n_particles)])

            # Pembaruan pbest
            better = costs < pbest_cost
            pbest_pos[better]  = pos[better].copy()
            pbest_cost[better] = costs[better]

            # Pembaruan gbest
            best_idx = np.argmin(pbest_cost)
            if pbest_cost[best_idx] < gbest_cost:
                gbest_cost = pbest_cost[best_idx]
                gbest_pos  = pbest_pos[best_idx].copy()

            history.append(gbest_cost)

        return gbest_pos, gbest_cost, history


class StandardPSO:
    """
    Standard PSO (SPSO) sebagai pembanding.
    w=0.729, c1=c2=1.494 (Kennedy & Eberhart, 1995)
    """
    def __init__(self, n_particles=30, max_iter=100,
                 w=0.729, c1=1.494, c2=1.494, bounds=None):
        self.n_particles = n_particles
        self.max_iter = max_iter
        self.w = w;  self.c1 = c1;  self.c2 = c2
        self.bounds = bounds

    def optimize(self, fitness_func, n_dim):
        bounds = self.bounds or [(-100, 100)] * n_dim
        lb = np.array([b[0] for b in bounds])
        ub = np.array([b[1] for b in bounds])
        v_max = 0.2 * (ub - lb)

        pos   = lb + np.random.rand(self.n_particles, n_dim) * (ub - lb)
        vel   = np.zeros((self.n_particles, n_dim))
        costs = np.array([fitness_func(pos[i]) for i in range(self.n_particles)])

        pbest_pos  = pos.copy()
        pbest_cost = costs.copy()
        gbest_idx  = np.argmin(pbest_cost)
        gbest_pos  = pbest_pos[gbest_idx].copy()
        gbest_cost = pbest_cost[gbest_idx]
        history    = [gbest_cost]

        for _ in range(self.max_iter):
            r1 = np.random.rand(self.n_particles, n_dim)
            r2 = np.random.rand(self.n_particles, n_dim)
            vel = (self.w * vel
                   + self.c1 * r1 * (pbest_pos - pos)
                   + self.c2 * r2 * (gbest_pos - pos))
            vel   = np.clip(vel, -v_max, v_max)
            pos   = np.clip(pos + vel, lb, ub)
            costs = np.array([fitness_func(pos[i]) for i in range(self.n_particles)])
            better = costs < pbest_cost
            pbest_pos[better]  = pos[better].copy()
            pbest_cost[better] = costs[better]
            best_idx = np.argmin(pbest_cost)
            if pbest_cost[best_idx] < gbest_cost:
                gbest_cost = pbest_cost[best_idx]
                gbest_pos  = pbest_pos[best_idx].copy()
            history.append(gbest_cost)

        return gbest_pos, gbest_cost, history


print('ADIWAKO dan StandardPSO berhasil didefinisikan!')
print('Perbaikan: |pbest_cost| sebagai penyebut mu + mu clipping [0,1] + velocity clamping')

---
## BAGIAN 2: Benchmark Functions Baru
Dipilih dari **Top-25** Naser et al. (2024) — belum digunakan di Sekyere et al. (2024).

| # | Fungsi | Tipe | Dim | Range | f* |
|---|---|---|---|---|---|
| 1 | Rastrigin | Multimodal | 2D & 20D | [-5.12, 5.12] | 0 |
| 2 | Styblinski-Tang | Multimodal | 2D | [-5, 5] | -78.332 |
| 3 | Levy N.13 | Multimodal | 2D | [-10, 10] | 0 |
| 4 | Schwefel 2.22 | Unimodal | 20D | [-10, 10] | 0 |
| 5 | Michalewicz | Multimodal | 10D | [0, π] | -9.66 |
| 6 | Zakharov | Unimodal | 20D | [-5, 10] | 0 |

In [ ]:
def rastrigin(x):
    """Naser et al. (2024) Eq.(213) | Multimodal | Range:[-5.12,5.12] | f*=0"""
    x = np.asarray(x, dtype=float)
    return 10 * len(x) + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

def styblinski_tang(x):
    """Naser et al. (2024) Eq.(270) | Multimodal | Range:[-5,5] | f*=-78.332 (2D)"""
    x = np.asarray(x, dtype=float)
    return 0.5 * np.sum(x**4 - 16 * x**2 + 5 * x)

def levy_n13(x):
    """Naser et al. (2024) Top-25 | Multimodal | Range:[-10,10] | f*=0"""
    x = np.asarray(x, dtype=float)
    x1, x2 = x[0], x[1]
    return (np.sin(3 * np.pi * x1)**2
            + (x1 - 1)**2 * (1 + np.sin(3 * np.pi * x2)**2)
            + (x2 - 1)**2 * (1 + np.sin(2 * np.pi * x2)**2))

def schwefel_2_22(x):
    """Naser et al. (2024) Eq.(244) | Unimodal | Range:[-10,10] | f*=0"""
    x = np.asarray(x, dtype=float)
    return np.sum(np.abs(x)) + np.prod(np.abs(x))

def michalewicz(x, m=10):
    """Naser et al. (2024) Eq.(164) | Multimodal | Range:[0,pi] | f*(10D)=-9.66"""
    x = np.asarray(x, dtype=float)
    i = np.arange(1, len(x) + 1)
    return -np.sum(np.sin(x) * (np.sin(i * x**2 / np.pi))**(2 * m))

def zakharov(x):
    """Naser et al. (2024) Eq.(311) | Unimodal | Range:[-5,10] | f*=0"""
    x = np.asarray(x, dtype=float)
    i = np.arange(1, len(x) + 1)
    s2 = np.sum(0.5 * i * x)
    return np.sum(x**2) + s2**2 + s2**4

BENCHMARK_CONFIG = [
    {'name':'Rastrigin (2D)',       'func':rastrigin,      'n_dim':2,
     'bounds':[(-5.12, 5.12)]*2,   'optimum':0.0,   'type':'Multimodal'},
    {'name':'Rastrigin (20D)',      'func':rastrigin,      'n_dim':20,
     'bounds':[(-5.12, 5.12)]*20,  'optimum':0.0,   'type':'Multimodal'},
    {'name':'Styblinski-Tang (2D)', 'func':styblinski_tang,'n_dim':2,
     'bounds':[(-5, 5)]*2,         'optimum':-78.332,'type':'Multimodal'},
    {'name':'Levy N.13 (2D)',       'func':levy_n13,       'n_dim':2,
     'bounds':[(-10, 10)]*2,       'optimum':0.0,   'type':'Multimodal'},
    {'name':'Schwefel 2.22 (20D)',  'func':schwefel_2_22,  'n_dim':20,
     'bounds':[(-10, 10)]*20,      'optimum':0.0,   'type':'Unimodal'},
    {'name':'Michalewicz (10D)',    'func':michalewicz,    'n_dim':10,
     'bounds':[(0, np.pi)]*10,     'optimum':-9.66, 'type':'Multimodal'},
    {'name':'Zakharov (20D)',       'func':zakharov,       'n_dim':20,
     'bounds':[(-5, 10)]*20,       'optimum':0.0,   'type':'Unimodal'},
]

print('Benchmark functions berhasil didefinisikan!')
for b in BENCHMARK_CONFIG:
    print(f"  {b['name']:25s} | {b['type']:10s} | dim={b['n_dim']:2d} | f*={b['optimum']}")

---
## BAGIAN 3: Eksperimen Benchmark — Validasi ADIWAKO

In [ ]:
def run_benchmark(benchmark_list, n_runs=10, n_particles=30, max_iter=100):
    results = []
    for bench in benchmark_list:
        print(f"\n{'='*55}")
        print(f"Fungsi : {bench['name']} | {bench['type']} | dim={bench['n_dim']}")
        print(f"{'='*55}")
        for algo_name, AlgoClass, kw in [
            ('ADIWAKO',      ADIWAKO,     {'W_max':1.0,'W_min':0.1,'C_max':5.0,'C_min':2.0}),
            ('Standard PSO', StandardPSO, {'w':0.729,'c1':1.494,'c2':1.494}),
        ]:
            run_costs, run_hists = [], []
            for run in range(n_runs):
                np.random.seed(run * 7 + 13)
                algo = AlgoClass(n_particles=n_particles, max_iter=max_iter,
                                 bounds=bench['bounds'], **kw)
                _, best, hist = algo.optimize(bench['func'], bench['n_dim'])
                run_costs.append(best)
                run_hists.append(hist)

            results.append({
                'Function'   : bench['name'],
                'Type'       : bench['type'],
                'Algorithm'  : algo_name,
                'Optimum'    : np.min(run_costs),
                'Mean'       : np.mean(run_costs),
                'Std'        : np.std(run_costs),
                'Theoretical': bench['optimum'],
                'History'    : np.mean(run_hists, axis=0),
            })
            print(f"  {algo_name:12s} | Optimum:{np.min(run_costs):12.4e} "
                  f"| Mean:{np.mean(run_costs):12.4e} | Std:{np.std(run_costs):12.4e}")

    return results

print('Memulai eksperimen benchmark (n_runs=10, n_particles=30, max_iter=100)...')
benchmark_results = run_benchmark(BENCHMARK_CONFIG, n_runs=10)
print('\nEksperimen benchmark selesai!')

In [ ]:
rows = []
for r in benchmark_results:
    rows.append({
        'Fungsi'    : r['Function'],
        'Tipe'      : r['Type'],
        'Algoritma' : r['Algorithm'],
        'Optimum'   : f"{r['Optimum']:.4e}",
        'Mean'      : f"{r['Mean']:.4e}",
        'Std Dev'   : f"{r['Std']:.4e}",
        'Teoritik'  : r['Theoretical'],
    })
df_bench = pd.DataFrame(rows)
print('=== TABEL HASIL BENCHMARK ===')
print(df_bench.to_string(index=False))

In [ ]:
n_funcs = len(BENCHMARK_CONFIG)
ncols   = 3
nrows   = int(np.ceil(n_funcs / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows))
axes = axes.flatten()
col  = {'ADIWAKO':'#E63946', 'Standard PSO':'#457B9D'}

for idx, bench in enumerate(BENCHMARK_CONFIG):
    ax = axes[idx]
    for r in benchmark_results:
        if r['Function'] == bench['name']:
            hist = np.array(r['History'])
            # shift supaya bisa di-log (untuk fungsi dengan nilai negatif)
            shift = abs(min(0, hist.min())) + 1e-300
            ax.semilogy(hist + shift,
                        label=r['Algorithm'],
                        color=col.get(r['Algorithm'], 'gray'),
                        linewidth=2)
    ax.set_title(f"{bench['name']}\n({bench['type']})", fontsize=9, fontweight='bold')
    ax.set_xlabel('Iterasi')
    ax.set_ylabel('Best Cost (log)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for i in range(n_funcs, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Convergence Curves: ADIWAKO vs Standard PSO\n'
             'Benchmark Functions Tambahan (Naser et al., 2024)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('convergence_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Convergence curves tersimpan: convergence_benchmark.png')

In [ ]:
# Ranking per fungsi (lebih kecil Mean = lebih baik, kecuali Michalewicz & Styblinski)
print('\n=== RANKING PER FUNGSI ===')
print(f'{"Fungsi":<25} {"Pemenang":<15} {"Selisih Mean"}')
print('-' * 60)

fn_names = list(dict.fromkeys(r['Function'] for r in benchmark_results))
adiwako_wins = 0
pso_wins     = 0

for fn in fn_names:
    data = {r['Algorithm']: r for r in benchmark_results if r['Function'] == fn}
    if 'ADIWAKO' not in data or 'Standard PSO' not in data:
        continue
    m_adi = data['ADIWAKO']['Mean']
    m_pso = data['Standard PSO']['Mean']
    # Untuk Michalewicz & Styblinski: lebih negatif = lebih baik
    if data['ADIWAKO']['Theoretical'] < 0:
        winner = 'ADIWAKO' if m_adi < m_pso else 'Standard PSO'
    else:
        winner = 'ADIWAKO' if m_adi < m_pso else 'Standard PSO'
    if winner == 'ADIWAKO':
        adiwako_wins += 1
    else:
        pso_wins += 1
    selisih = abs(m_adi - m_pso)
    print(f'{fn:<25} {winner:<15} {selisih:.4e}')

print(f'\nTotal Kemenangan:')
print(f'  ADIWAKO      : {adiwako_wins} fungsi')
print(f'  Standard PSO : {pso_wins} fungsi')

---
## BAGIAN 4: Eksperimen SVM
### 4.1 Load Dataset

In [ ]:
import urllib.request, zipfile, io

DATASETS = {}

# ── Dataset 1: Bank Marketing ──────────────────────────────
print('Mengunduh Bank Marketing Dataset...')
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip'
try:
    with urllib.request.urlopen(url, timeout=30) as resp:
        z = zipfile.ZipFile(io.BytesIO(resp.read()))
        with z.open('bank-full.csv') as f:
            df_bank = pd.read_csv(f, sep=';')
    le = LabelEncoder()
    df_tmp = df_bank.copy()
    for col in df_tmp.select_dtypes(include='object').columns:
        df_tmp[col] = le.fit_transform(df_tmp[col])
    X = StandardScaler().fit_transform(df_tmp.drop('y', axis=1).values)
    y = df_tmp['y'].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                           random_state=42, stratify=y)
    DATASETS['Bank Marketing'] = (Xtr, Xte, ytr, yte)
    print(f'  Bank Marketing: {df_bank.shape} | Train={Xtr.shape} | Kelas={np.bincount(ytr)}')
except Exception as e:
    print(f'  Gagal download: {e}')
    print('  Silakan upload bank-full.csv secara manual ke Colab.')

# ── Dataset 2: Breast Cancer Wisconsin ────────────────────
data_bc = load_breast_cancer()
X = StandardScaler().fit_transform(data_bc.data)
y = data_bc.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                       random_state=42, stratify=y)
DATASETS['Breast Cancer'] = (Xtr, Xte, ytr, yte)
print(f'  Breast Cancer : {data_bc.data.shape} | Train={Xtr.shape} | Kelas={np.bincount(ytr)}')

# ── Dataset 3: Pima Indians Diabetes ──────────────────────
pima_url = ('https://raw.githubusercontent.com/jbrownlee/Datasets/'
            'master/pima-indians-diabetes.data.csv')
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
        'Insulin','BMI','DiabetesPedigree','Age','Outcome']
try:
    df_pima = pd.read_csv(pima_url, names=cols)
    X = StandardScaler().fit_transform(df_pima.drop('Outcome', axis=1).values)
    y = df_pima['Outcome'].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                           random_state=42, stratify=y)
    DATASETS['Pima Diabetes'] = (Xtr, Xte, ytr, yte)
    print(f'  Pima Diabetes : {df_pima.shape} | Train={Xtr.shape} | Kelas={np.bincount(ytr)}')
except Exception as e:
    print(f'  Pima gagal: {e}')

print(f'\nDataset siap: {list(DATASETS.keys())}')

### 4.2 Fungsi Fitness & Evaluasi SVM

In [ ]:
# Ruang pencarian hyperparameter dalam skala log10
# C     : 10^-3  s.d. 10^3   → bounds log10: [-3, 3]
# gamma : 10^-5  s.d. 10^2   → bounds log10: [-5, 2]
SVM_BOUNDS = [(-3, 3), (-5, 2)]

def svm_fitness(params, X_train, y_train, cv=5):
    """Fitness = 1 - rata-rata akurasi CV (minimisasi)."""
    C     = float(np.clip(10 ** params[0], 1e-3, 1e5))
    gamma = float(np.clip(10 ** params[1], 1e-5, 1e3))
    svm   = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)
    skf   = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    score = cross_val_score(svm, X_train, y_train,
                            cv=skf, scoring='accuracy').mean()
    return 1.0 - score

def evaluate_svm(C, gamma, Xtr, ytr, Xte, yte):
    """Latih SVM final dan evaluasi metrik lengkap."""
    svm = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)
    svm.fit(Xtr, ytr)
    yp  = svm.predict(Xte)
    return {
        'C'        : C,
        'gamma'    : gamma,
        'Accuracy' : accuracy_score(yte, yp),
        'Precision': precision_score(yte, yp, average='weighted', zero_division=0),
        'Recall'   : recall_score(yte, yp,    average='weighted', zero_division=0),
        'F1-Score' : f1_score(yte, yp,        average='weighted', zero_division=0),
        'CM'       : confusion_matrix(yte, yp),
    }

print('Fungsi fitness dan evaluasi SVM siap!')
print('Ruang pencarian: C=[10^-3,10^3], gamma=[10^-5,10^2] (log10 scale)')

### 4.3 Eksperimen: SVM Baseline, SVM+PSO, SVM+ADIWAKO

In [ ]:
all_svm_results = {}

for ds_name, (Xtr, Xte, ytr, yte) in DATASETS.items():
    print(f'\n{"="*55}')
    print(f'DATASET: {ds_name}')
    print(f'{"="*55}')
    res = {}

    # ── 1. SVM Baseline ────────────────────────────────────
    print('[1] SVM Baseline (C=1.0, gamma=0.1, default sklearn)')
    r = evaluate_svm(1.0, 0.1, Xtr, ytr, Xte, yte)
    res['SVM Baseline'] = r
    print(f'    Accuracy={r["Accuracy"]:.4f} | F1={r["F1-Score"]:.4f}')

    fn = lambda p: svm_fitness(p, Xtr, ytr)

    # ── 2. SVM + PSO Standar ───────────────────────────────
    print('[2] SVM + PSO Standar')
    pso = StandardPSO(n_particles=20, max_iter=50, bounds=SVM_BOUNDS)
    bp_pso, _, hist_pso = pso.optimize(fn, n_dim=2)
    C_pso   = float(10 ** bp_pso[0])
    g_pso   = float(10 ** bp_pso[1])
    r = evaluate_svm(C_pso, g_pso, Xtr, ytr, Xte, yte)
    res['SVM + PSO'] = r
    print(f'    C={C_pso:.6f} | gamma={g_pso:.8f}')
    print(f'    Accuracy={r["Accuracy"]:.4f} | F1={r["F1-Score"]:.4f}')

    # ── 3. SVM + ADIWAKO ───────────────────────────────────
    print('[3] SVM + ADIWAKO')
    adi = ADIWAKO(n_particles=20, max_iter=50,
                  W_max=1.0, W_min=0.1, C_max=5.0, C_min=2.0,
                  bounds=SVM_BOUNDS)
    bp_adi, _, hist_adi = adi.optimize(fn, n_dim=2)
    C_adi   = float(10 ** bp_adi[0])
    g_adi   = float(10 ** bp_adi[1])
    r = evaluate_svm(C_adi, g_adi, Xtr, ytr, Xte, yte)
    res['SVM + ADIWAKO'] = r
    print(f'    C={C_adi:.6f} | gamma={g_adi:.8f}')
    print(f'    Accuracy={r["Accuracy"]:.4f} | F1={r["F1-Score"]:.4f}')

    res['_hist_pso'] = hist_pso
    res['_hist_adi'] = hist_adi
    all_svm_results[ds_name] = res

print('\nSemua eksperimen SVM selesai!')

### 4.4 Tabel Perbandingan & Visualisasi

In [ ]:
rows = []
for ds, ds_res in all_svm_results.items():
    for model in ['SVM Baseline', 'SVM + PSO', 'SVM + ADIWAKO']:
        if model in ds_res:
            r = ds_res[model]
            rows.append({
                'Dataset'  : ds,
                'Model'    : model,
                'C'        : f"{r['C']:.4f}",
                'Gamma'    : f"{r['gamma']:.6f}",
                'Accuracy' : f"{r['Accuracy']:.4f}",
                'Precision': f"{r['Precision']:.4f}",
                'Recall'   : f"{r['Recall']:.4f}",
                'F1-Score' : f"{r['F1-Score']:.4f}",
            })
df_compare = pd.DataFrame(rows)
print('=== TABEL PERBANDINGAN KINERJA SVM ===')
print(df_compare.to_string(index=False))

In [ ]:
ds_names = list(all_svm_results.keys())
n_ds     = len(ds_names)
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models   = ['SVM Baseline', 'SVM + PSO', 'SVM + ADIWAKO']
colors_b = ['#6c757d', '#457B9D', '#E63946']

fig = plt.figure(figsize=(16, 5.5 * n_ds))
gs  = gridspec.GridSpec(n_ds, 2, figure=fig, hspace=0.55, wspace=0.3)

for di, ds_name in enumerate(ds_names):
    ds_res = all_svm_results[ds_name]

    # Bar chart
    ax1 = fig.add_subplot(gs[di, 0])
    x = np.arange(len(metrics));  w = 0.25
    for mi, (mdl, clr) in enumerate(zip(models, colors_b)):
        if mdl in ds_res:
            vals = [ds_res[mdl][met] for met in metrics]
            bars = ax1.bar(x + mi * w, vals, w, label=mdl, color=clr, alpha=0.85)
            for bar, v in zip(bars, vals):
                ax1.text(bar.get_x() + bar.get_width() / 2,
                         bar.get_height() + 0.002,
                         f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax1.set_title(f'{ds_name}\nPerbandingan Metrik', fontweight='bold')
    ax1.set_xticks(x + w);  ax1.set_xticklabels(metrics)
    ax1.set_ylim(0, 1.12);  ax1.set_ylabel('Nilai')
    ax1.legend(fontsize=8);  ax1.grid(axis='y', alpha=0.3)

    # Convergence curve
    ax2 = fig.add_subplot(gs[di, 1])
    if '_hist_pso' in ds_res:
        ax2.plot(ds_res['_hist_pso'], label='PSO Standar', color='#457B9D', lw=2)
    if '_hist_adi' in ds_res:
        ax2.plot(ds_res['_hist_adi'], label='ADIWAKO',     color='#E63946', lw=2)
    ax2.set_title(f'{ds_name}\nKurva Konvergensi', fontweight='bold')
    ax2.set_xlabel('Iterasi');  ax2.set_ylabel('1 - Akurasi CV (Fitness)')
    ax2.legend(fontsize=9);  ax2.grid(alpha=0.3)

plt.suptitle('Hasil Eksperimen SVM: Baseline vs PSO vs ADIWAKO',
             fontsize=14, fontweight='bold')
plt.savefig('svm_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot tersimpan: svm_results.png')

### 4.5 Confusion Matrix

In [ ]:
ds_names = list(all_svm_results.keys())
n_ds     = len(ds_names)
models   = ['SVM Baseline', 'SVM + PSO', 'SVM + ADIWAKO']
cmaps    = ['Blues', 'YlOrBr', 'Reds']

fig, axes = plt.subplots(n_ds, 3, figsize=(13, 4.5 * n_ds))
if n_ds == 1:
    axes = axes[np.newaxis, :]

for di, ds_name in enumerate(ds_names):
    for mi, (mdl, cmap) in enumerate(zip(models, cmaps)):
        ax = axes[di, mi]
        if mdl not in all_svm_results[ds_name]:
            ax.set_visible(False);  continue
        r  = all_svm_results[ds_name][mdl]
        cm = r['CM']
        im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
        plt.colorbar(im, ax=ax)
        thresh = cm.max() / 2.0
        for ii in range(cm.shape[0]):
            for jj in range(cm.shape[1]):
                ax.text(jj, ii, str(cm[ii, jj]), ha='center', va='center',
                        color='white' if cm[ii, jj] > thresh else 'black',
                        fontsize=11, fontweight='bold')
        ax.set_title(f'{ds_name}\n{mdl}\nAcc={r["Accuracy"]:.4f}',
                     fontsize=9, fontweight='bold')
        ax.set_xlabel('Prediksi');  ax.set_ylabel('Aktual')

plt.suptitle('Confusion Matrix — Semua Dataset & Model',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrices tersimpan: confusion_matrices.png')

---
## BAGIAN 5: Ringkasan Hasil

In [ ]:
print('=' * 65)
print('RINGKASAN HASIL EKSPERIMEN SVM')
print('=' * 65)
for ds_name, ds_res in all_svm_results.items():
    print(f'\nDataset: {ds_name}')
    print(f'{"Model":<20} {"C":>10} {"Gamma":>12} {"Acc":>8} {"F1":>8}')
    print('-' * 62)
    for model in ['SVM Baseline', 'SVM + PSO', 'SVM + ADIWAKO']:
        if model in ds_res:
            r = ds_res[model]
            print(f'{model:<20} {r["C"]:>10.4f} {r["gamma"]:>12.6f} '
                  f'{r["Accuracy"]:>8.4f} {r["F1-Score"]:>8.4f}')
print('\nFile output:')
print('  convergence_benchmark.png | svm_results.png | confusion_matrices.png')